In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
import random
from pyproj import Proj, Transformer
import math
from scipy.spatial.kdtree import KDTree
import datetime
import itertools
from netCDF4 import Dataset
import scipy.stats
import glob
import os
import tqdm
import warnings
from regrid import regrid
warnings.filterwarnings("ignore")

In [3]:
def lonlat_to_xy(coords_1, coords_2, hemisphere, inverse=False):

    """Converts between longitude/latitude and EASE xy coordinates.
 
    Args:
        lon (float): WGS84 longitude
        lat (float): WGS84 latitude
        hemisphere (string): 'n' or 's'
        inverse (bool): if true, converts xy to lon/lat
 
    Returns:
        tuple: pair of xy or lon/lat values
    """

    EASE_Proj = {'n': 'EPSG:3408',
                 's': 'EPGS:3409'}
    
    WGS_Proj = 'EPSG:4326'
    
    for coords in [coords_1, coords_2]: assert isinstance(coords,(np.ndarray,list))

    if inverse == False: # lonlat to xy
        
        lon, lat = coords_1, coords_2
        
        transformer = Transformer.from_crs(WGS_Proj, EASE_Proj[hemisphere])
        
        x, y = transformer.transform(lat, lon)
        
        return (x, y)

    else: # xy to lonlat
        
        x, y = coords_1, coords_2
        
        transformer = Transformer.from_crs(EASE_Proj[hemisphere], WGS_Proj)
        
        lat, lon = transformer.transform(x, y)
        
        return (lon, lat)

In [4]:
datapath_aux = '/Users/carmennab/Dropbox/pb_new/data/roughness/auxiliary/'
ease_lons = np.load(f'{datapath_aux}1.1km_lon_cent.npy')
ease_lats = np.load(f'{datapath_aux}1.1km_lat_cent.npy')

# convert into x,y:
ease_x, ease_y = lonlat_to_xy(ease_lons,ease_lats,hemisphere='n')

# create KDTree to find EASE grid cell closest to buoy:
tree = KDTree(list(zip(ease_x.ravel(),ease_y.ravel())))

In [5]:
### calculate distance to land for each grid cell
region_file = Dataset(f'{datapath_aux}NSIDC_Regions_Mask_ease.nc')
region_mask_25 = np.array(region_file['mask'])
lon_25 = np.array(region_file['longitude'])
lat_25 = np.array(region_file['latitude'])
region_mask = regrid(region_mask_25,lon_25,lat_25,ease_lons,ease_lats)

In [6]:
ease_xs, ease_ys = lonlat_to_xy(ease_lons,ease_lats,hemisphere='n')
land_xs = ease_xs[~np.isin(region_mask,4)] ; land_ys = ease_ys[~np.isin(region_mask,4)]
land_tree = KDTree(list(zip(land_xs.ravel(),land_ys.ravel())))

In [7]:
min_ds = np.full(ease_xs.shape,1)
for i, j in itertools.product(np.arange(ease_xs.shape[0]),np.arange(ease_xs.shape[1])):
    ease_x = ease_xs[i,j] ; ease_y = ease_ys[i,j]
    ind2d = np.unravel_index(land_tree.query([ease_x,ease_y])[1], land_xs.shape)
    min_ds[i,j] = math.sqrt((land_xs[ind2d]-ease_x)**2 + (land_ys[ind2d]-ease_y)**2)

In [8]:
datapath = '/Users/carmennab/Dropbox/pb_new/'
### load steps lengths
steps = pd.read_csv(f'{datapath}scripts/SSF/step_distances.csv')
steps['month'] = [int(date[5:7]) for date in steps['date']]

### load turning angles 
angles = pd.read_csv(f'{datapath}scripts/SSF/turning_angles.csv')
angles['month'] = [int(date[5:7]) for date in angles['date']]

In [9]:
def gamma_random_sample():
    """Yields a list of random numbers following a gamma distribution defined by mean and variance"""
    mean = np.mean(bear_steps)
    variance = np.var(bear_steps)
    g_alpha = mean*mean/variance
    g_beta = mean/variance
    for i in range(tries+1):
        yield random.gammavariate(g_alpha,1/g_beta)
        
def vm_random_sample():
    """Yields a list of random numbers following a vonmises distribution defined by mu and kappa"""
    kappa, loc, scale = scipy.stats.vonmises.fit(bear_angles) 
    mu = 0
    for i in range(tries+1):
        yield random.vonmisesvariate(mu,kappa)

In [10]:
def turning_angle(xs,ys,count):
    a = [xs[count-2],ys[count-2]] 
    b = [xs[count-1],ys[count-1]] 
    c = [xs[count],ys[count]]

    ab = np.mod((180/math.pi)*math.atan2(b[0]-a[0],
                                         b[1]-a[1]),
                360)

    bc = np.mod((180/math.pi)*math.atan2(c[0]-b[0], #delta_x
                                         c[1]-b[1]), #delta_y
                360)

    angle = np.mod(bc-ab,360) # +ve when clockwise

    if angle > 180:
        angle = angle - 360
        
    return angle

In [11]:
sit_datapath = '/Users/carmennab/Dropbox/pb_new/data/SIT/CS2SMOS/1.1km/'
sd_datapath = '/Users/carmennab/Dropbox/pb_new/data/SIT/CS2SMOS/1.1km/'
sic_datapath = '/Users/carmennab/Dropbox/pb_new/data/SIC/1.1km/'
sr_datapath = '/Users/carmennab/Dropbox/pb_new/data/roughness/daily_mean_roughness/1.1km_ease/'

In [12]:
### load collar data
datapath = '/Users/carmennab/Dropbox/pb_new/'
collar_data = pd.read_csv(f'{datapath}data/bears/data_combined.csv')
collar_data

,ID,longitude,latitude,time,type,date,sex,age,x,y,ind2d,SD,SIT,SR,SIC,d_to_land
0,X12730,-93.719516,57.635350,5:01:27,collar,2012-09-29,F,Adult,-3.543776e+06,230377.895490,"(70, 43)",NaN,NaN,NaN,NaN,0.0
1,X12730,-93.719430,57.635179,9:00:53,collar,2012-09-29,F,Adult,-3.543795e+06,230373.760985,"(70, 43)",NaN,NaN,NaN,NaN,0.0
2,X12730,-93.719430,57.635350,13:01:04,collar,2012-09-29,F,Adult,-3.543777e+06,230372.576341,"(70, 43)",NaN,NaN,NaN,NaN,0.0
3,X12730,-93.719430,57.635264,17:02:27,collar,2012-09-29,F,Adult,-3.543786e+06,230373.172127,"(70, 43)",NaN,NaN,NaN,NaN,0.0
4,X12730,-93.719430,57.635264,21:00:52,collar,2012-09-29,F,Adult,-3.543786e+06,230373.172127,"(70, 43)",NaN,NaN,NaN,NaN,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
390339,X33697,-89.668000,56.690000,1:30:52,collar,2022-09-01,M,Subadult,-3.652031e+06,-21161.902714,"(60, 38)",NaN,NaN,NaN,NaN,0.0
390340,X33697,-89.666000,56.690000,1:38:07,collar,2022-09-01,M,Subadult,-3.652030e+06,-21289.382616,"(60, 38)",NaN,NaN,NaN,NaN,0.0
390341,X33697,-89.655000,56.697000,1:54:47,collar,2022-09-01,M,Subadult,-3.651280e+06,-21986.031223,"(60, 38)",NaN,NaN,NaN,NaN,0.0
390342,X33697,-89.684000,56.693000,2:49:10,collar,2022-09-01,M,Subadult,-3.651717e+06,-20140.299905,"(60, 38)",NaN,NaN,NaN,NaN,0.0


In [13]:
n = 100 ### specify number of random steps 
tries = 1000 ### specify number of tries

### generate random steps for each bear 

In [14]:
for age,sex,class_name in zip(['Adult','Adult','Subadult'],[['F'],['M'],['F','M']],['Adult_F','Adult_M','Subadult']):
    class_data = collar_data[(collar_data['age']==age)& (np.isin(collar_data['sex'],sex))]
    datapath_out = f'/Users/carmennab/Dropbox/pb_new/scripts/SSF/random_steps/1.1km/{class_name}/'

    for ID in tqdm.tqdm(set(class_data['ID'])): ### loop through each individual bear
        filepath_out = f'{datapath_out}{ID}.csv'
        random_df = []

        bear_steps = np.array(steps[steps['ID']==ID]['step_distance (m)'])
        g_steps = [int(i) for i in gamma_random_sample()]
        bear_angles = np.array(angles[angles['ID']==ID]['angle'])
        vm_angles = [(int(i)*180/math.pi) for i in vm_random_sample()] # fit vonmises distribution to turning angles
        bear_data = collar_data[collar_data['ID']==ID].drop(columns=['ID'])

        ### select step 
        xs = np.array(bear_data['x'])
        ys = np.array(bear_data['y'])
        dates = [(datetime.datetime.strptime(d,'%Y-%m-%d')).date() for d in bear_data['date']]
        str_dates = [date.strftime('%Y%m%d') for date in dates]

        for c in np.arange(0,len(xs)-1):
            points_df = []
            sitpath = f'{sit_datapath}{str_dates[c]}.nc'
            sdpath = f'{sd_datapath}{str_dates[c]}.nc'
            sicpath = f'{sic_datapath}{str_dates[c]}.nc'
            srpath = f'{sr_datapath}{str_dates[c][0:4]}/{str_dates[c]}.nc'
            if (os.path.exists(sdpath) == True) & (os.path.exists(sicpath) == True) & (os.path.exists(srpath) == True):
                all_sits = np.array(Dataset(sitpath)['sea_ice_thickness'])
                all_sds = np.array(Dataset(sitpath)['snow_depth'])
                all_sics = np.array(Dataset(sicpath)['sic']) 
                all_srs = np.array(Dataset(srpath)['roughness']) 

                ### load start point
                x1 = xs[c] ; y1 = ys[c] 
                ind2d = np.unravel_index(tree.query([x1,y1])[1], (ease_lats.shape[0],ease_lats.shape[1]))

                ### load end point
                x2 = xs[c+1] ;  y2 = ys[c+1]

                ### calculate step distance and turning angle
                d = int(math.sqrt((x2-x1)**2 + (y2-y1)**2))
                angle = turning_angle(xs,ys,c)
                ind2d_i = np.unravel_index(tree.query([x2,y2])[1], (ease_lats.shape[0],ease_lats.shape[1])) 
                points_df.append({'ID':ID,'date':dates[c],'step_id':c,'ind2d':ind2d,'sit':all_sits[ind2d],
                                  'sic':all_sics[ind2d],'sr':all_srs[ind2d],'sd':all_sds[ind2d],'case':1,
                                  'd_to_land':min_ds[ind2d],'step_length':d,'turning_angle':angle})

                ### load random points
                p = 3 ; t = 0 
                while (p < (n+3)) & (t < tries): ### keep trying until n random points reached, or until specified # of tries
                    d_r = np.random.choice(g_steps) # select random step distance
                    angle_r = np.random.choice(vm_angles) # select random angle

                    ### calculate random x2 and y2
                    x2_r = x1 + (d_r * math.cos(angle_r * math.pi / 180))
                    y2_r = y1 + (d_r * math.sin(angle_r * math.pi / 180))

                    ind2d_r = np.unravel_index(tree.query([x2_r,y2_r])[1], (ease_lats.shape[0],ease_lats.shape[1])) 

                    ### save if ice at random location, otherwise try again
                    if (~np.isnan(all_sits[ind2d_r])) & (~np.isnan(all_sics[ind2d_r])):
                        name = f'Y{p}'
                        p+=1
                        points_df.append({'ID':ID,'date':dates[c],'step_id':c,'ind2d':ind2d,'sit':all_sits[ind2d_r],
                                          'sic':all_sics[ind2d_r],'sr':all_srs[ind2d_r],'sd':all_sds[ind2d_r],
                                          'd_to_land':min_ds[ind2d_r],'case':0,'step_length':d_r,
                                          'turning_angle':angle_r})
                    t += 1

                if p == (n+3): ### only use points where n random steps available 
                    for dic in points_df:
                        random_df.append(dic)

        if len(random_df)>0: 
            random_df = pd.DataFrame(random_df)
            random_df.to_csv(filepath_out,index=False) ### save steps for SSF
        else: pass

100%|███████████████████████████████████████████| 16/16 [00:10<00:00,  1.51it/s]
